In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
use catalog lendingclub;
create schema if not exists bronze;
use schema bronze;
select current_catalog(), current_schema();

In [0]:
loans_defaulter_df = spark.read.csv('/Volumes/lendingclub/storagelocation/landing/LoanDefaulter/', header = True, inferSchema = True)

In [0]:
display(loans_defaulter_df.head(5))

In [0]:
loans_defaulter_df.printSchema()

In [0]:
#observed that we can see some string values in delinq_2yrs ideally it should be float

display(loans_defaulter_df.select('delinq_2yrs').distinct())

In [0]:
loans_defaulter_df.groupby('delinq_2yrs').count().sort(desc('count')).show(40)

In [0]:
#renaming and schema enforcement
#delinq_2yrs enforced as float, so string values wil be come as null

defaulter_schema = '''member_id string, delinq_2yrs float, delinq_amount float, public_rec float, public_bankruptcies float, inquiry_6_months float,
total_recevied_date_fee float, months_since_last_delinq float, months_since_last_public_record float'''

In [0]:
loans_defaulter_df = spark.read\
.format('csv')\
.option('header', True)\
.schema(defaulter_schema)\
.load('/Volumes/lendingclub/storagelocation/landing/LoanDefaulter/')

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/raw/LoanDefaulter/", recurse=True)

In [0]:
loans_defaulter_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/raw/LoanDefaulter/')

In [0]:
%sql
create or replace table bronze.LoanDefaulter
as
select * 
from delta.`/Volumes/lendingclub/storagelocation/raw/LoanDefaulter/`

In [0]:
%sql
select * from bronze.loandefaulter limit 2